# 대용량 CSV 탐색 - DuckDB & Polars Lazy API

---

## 라이브러리 소개

### DuckDB

**DuckDB**는 분석용(OLAP) 인메모리 SQL 데이터베이스 엔진입니다. SQLite처럼 설치가 필요 없고 Python 패키지 하나로 바로 사용할 수 있습니다.

**핵심 특징:**
- **파일 직접 쿼리**: CSV, Parquet, JSON 파일을 메모리에 올리지 않고 SQL로 직접 읽음
- **벡터화 실행 엔진**: 컬럼 단위로 데이터를 배치 처리 → pandas 대비 10~100배 빠른 집계
- **스트리밍 처리**: 메모리보다 큰 파일도 처리 가능 (80GB 파일을 수백 MB RAM으로 쿼리)
- **멀티스레드**: CPU 코어를 자동으로 활용
- **glob 패턴**: `*.csv`로 여러 파일을 한 번에 쿼리
- **pandas 연동**: `.df()`로 결과를 즉시 DataFrame으로 변환

**언제 쓰나:**
- 파일을 전부 로드하기 전에 빠르게 구조/통계를 확인하고 싶을 때
- SQL 문법이 익숙하고 GROUP BY, JOIN 등 집계 작업을 할 때
- 여러 파일을 합쳐서 한 번에 쿼리할 때

```python
import duckdb
con = duckdb.connect()
con.execute("SELECT COUNT(*) FROM read_csv_auto('data.csv')").df()
```

---

### Polars

**Polars**는 Rust로 작성된 고성능 DataFrame 라이브러리입니다. pandas의 대체재로 설계되었으며, 특히 대용량 데이터에서 압도적으로 빠릅니다.

**핵심 특징:**
- **Lazy API (`scan_csv`)**: 실행 계획만 만들고, 실제로 필요할 때만 데이터를 읽음 → 필요한 컬럼/행만 I/O
- **쿼리 최적화**: `.explain()`으로 실행 계획 확인, 불필요한 연산 자동 제거
- **Streaming 모드 (`.collect(streaming=True)`)**: 파일을 청크 단위로 처리하여 메모리 초과 방지
- **`sink_parquet()`**: 결과를 메모리에 올리지 않고 바로 Parquet 파일로 저장
- **Apache Arrow 기반**: 컬럼형 메모리 포맷 → 집계·필터 연산이 pandas보다 5~50배 빠름
- **멀티스레드**: GIL 없이 모든 CPU 코어 활용

**Lazy vs Eager 차이:**

| | Eager (pandas 방식) | Lazy (Polars 방식) |
|--|--|--|
| 실행 시점 | 코드 줄 즉시 | `.collect()` 호출 시 |
| 최적화 | 없음 | 자동 (불필요 컬럼 제거 등) |
| 대용량 처리 | 메모리 초과 위험 | Streaming으로 안전 |

**언제 쓰나:**
- 특정 컬럼만 선택해서 읽을 때 (시계열 컬럼 제외)
- 데이터를 변환해서 Parquet으로 저장할 때
- pandas 문법은 익숙하지만 더 빠른 처리가 필요할 때

```python
import polars as pl
(
    pl.scan_csv('data.csv')          # 읽기 안 함 (계획만)
    .select(['col1', 'col2'])         # 필요한 컬럼만
    .filter(pl.col('col1') > 0)       # 조건 필터
    .collect(streaming=True)          # 여기서 실제로 읽음
)
```

---

## 이 데이터셋

| 파일 | 크기 | 행 수 |
|------|------|-------|
| acn_all_sites.csv | ~80 GB | ~66,746 |
| caltech.csv | ~36 GB | ~31,425 |
| jpl.csv | ~43 GB | ~33,639 |
| office001.csv | ~2 GB | ~1,684 |

각 행에 `pilotSignal`, `chargingCurrent` 컬럼에 시계열 데이터가 통째로 문자열로 저장되어 있어 행 수 대비 용량이 큽니다.

## 전략
- **DuckDB**: 파일을 메모리에 올리지 않고 SQL로 직접 쿼리 → 가장 빠르고 메모리 효율적
- **Polars Lazy (scan_csv)**: 필요한 컬럼/행만 스캔하여 처리
- **Pandas chunksize**: 기존 방식, 시계열 컬럼 파싱 등 행 단위 처리가 필요할 때

In [ ]:
import duckdb
import polars as pl
import pandas as pd
import os

DATA_DIR = "acn_data"

# 파일 경로 정의
files = {
    "office001": f"{DATA_DIR}/office001.csv",   # ~2 GB  (탐색 시작점)
    "caltech":   f"{DATA_DIR}/caltech.csv",      # ~36 GB
    "jpl":       f"{DATA_DIR}/jpl.csv",          # ~43 GB
    "all_sites": f"{DATA_DIR}/acn_all_sites.csv", # ~80 GB
}

# 파일 크기 확인
for name, path in files.items():
    size_gb = os.path.getsize(path) / 1024**3
    print(f"{name:12s}: {size_gb:.2f} GB")

---
## 1. DuckDB — 메모리 없이 SQL로 즉시 탐색 (가장 추천)

DuckDB는 CSV를 **스트리밍**으로 읽기 때문에 80GB 파일도 SELECT 몇 줄로 바로 탐색 가능합니다.

In [ ]:
# DuckDB 연결 (in-memory)
con = duckdb.connect()

# 행 1개가 최대 ~100MB → max_line_size=200000000 필수
# 모든 read_csv_auto 호출에 공통으로 적용하기 위해 헬퍼 정의
def READ_CSV(path):
    return f"read_csv_auto('{path}', header=true, max_line_size=200000000)"

# ── 1-1. 컬럼명 & 타입 확인 ───────────────────────────────────────────
con.execute(f"DESCRIBE SELECT * FROM {READ_CSV(files['office001'])}").df()


In [ ]:
# ── 1-2. 행 수 / 기본 집계 (전체 파일 스캔, 메모리 최소) ─────────────────
con.execute(f"""
    SELECT
        COUNT(*)                  AS total_rows,
        COUNT(DISTINCT siteID)    AS unique_sites,
        COUNT(DISTINCT stationID) AS unique_stations,
        MIN(connectionTime)       AS first_connection,
        MAX(connectionTime)       AS last_connection,
        ROUND(SUM(kWhDelivered), 2) AS total_kWh,
        ROUND(AVG(kWhDelivered), 4) AS avg_kWh
    FROM {READ_CSV(files['office001'])}
""").df()

In [ ]:
# ── 1-3. 시계열 컬럼(pilotSignal, chargingCurrent) 제외하고 핵심 컬럼만 로드 ──
# → 이렇게 하면 메모리 사용량이 극적으로 줄어듦
df_meta = con.execute(f"""
    SELECT
        _id, userID, sessionID, stationID, spaceID, siteID, clusterID,
        connectionTime, disconnectTime, kWhDelivered, doneChargingTime, timezone
    FROM {READ_CSV(files['office001'])}
""").df()

print(f"메모리 사용량: {df_meta.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
df_meta.head()

In [ ]:
# ── 1-4. 스테이션별 충전량 집계 (SQL 집계 → 결과만 pandas로) ─────────────
con.execute(f"""
    SELECT
        stationID,
        COUNT(*) AS session_count,
        ROUND(SUM(kWhDelivered), 2) AS total_kWh,
        ROUND(AVG(kWhDelivered), 4) AS avg_kWh
    FROM {READ_CSV(files['office001'])}
    GROUP BY stationID
    ORDER BY total_kWh DESC
    LIMIT 10
""").df()

In [ ]:
# ── 1-5. 날짜별 세션 수 트렌드 ────────────────────────────────────────────
daily = con.execute(f"""
    SELECT
        CAST(connectionTime AS DATE) AS date,
        COUNT(*) AS sessions,
        ROUND(SUM(kWhDelivered), 2) AS total_kWh
    FROM {READ_CSV(files['office001'])}
    GROUP BY date
    ORDER BY date
""").df()

daily.plot(x='date', y=['sessions', 'total_kWh'], subplots=True, figsize=(12, 6), title='Daily Sessions & kWh')

---
## 2. DuckDB — 여러 파일 동시 쿼리 (glob 패턴)

전체 데이터셋을 하나의 쿼리로 처리할 수 있습니다.

In [ ]:
# ── 2-1. 사이트별 전체 통계 (caltech + jpl + office001 동시 쿼리) ──────────
# glob 패턴으로 여러 파일 동시 읽기 (acn_all_sites.csv 제외)
con.execute(f"""
    SELECT
        siteID,
        COUNT(*) AS sessions,
        ROUND(SUM(kWhDelivered), 2) AS total_kWh
    FROM read_csv_auto(
        ['{files['office001']}', '{files['caltech']}', '{files['jpl']}'],
        header=true,
        max_line_size=200000000,
        union_by_name=true
    )
    GROUP BY siteID
    ORDER BY total_kWh DESC
""").df()

---
## 3. Polars Lazy API — scan_csv (streaming)

Polars는 `scan_csv()`로 실행 계획만 만들고, `.collect()`나 `.sink_*()`를 부를 때만 실제로 읽습니다.  
필요한 컬럼/필터만 선언하면 **I/O 자체를 줄여줍니다.**

In [ ]:
# ── 3-1. Lazy 스캔 — 실행 계획 확인 (실제 읽기 전) ────────────────────────
lf = pl.scan_csv(
    files['office001'],
    encoding='utf8-lossy',
    infer_schema_length=1000,
)

# 시계열 컬럼 제외, 핵심 컬럼만 선택 → 실제로 해당 컬럼만 읽음
query = (
    lf.select([
        '_id', 'userID', 'sessionID', 'stationID',
        'siteID', 'connectionTime', 'disconnectTime',
        'kWhDelivered', 'doneChargingTime', 'timezone'
    ])
    .filter(pl.col('kWhDelivered') > 0)
)

# 실행 계획 출력 (읽기 전)
print(query.explain())

In [ ]:
# ── 3-2. 실제 수집 ─────────────────────────────────────────────────────
# streaming=True → v1.25 이후 deprecated, engine='streaming' 사용
df_pl = query.collect(engine='streaming')

print(df_pl.shape)
print(f"메모리: {df_pl.estimated_size('mb'):.1f} MB")
df_pl.head()

In [ ]:
# ── 3-3. Polars describe (빠른 통계) ──────────────────────────────────────
df_pl.describe()

In [ ]:
# ── 3-4. Polars groupby 집계 ──────────────────────────────────────────────
(
    df_pl
    .group_by('stationID')
    .agg([
        pl.len().alias('sessions'),
        pl.col('kWhDelivered').sum().alias('total_kWh'),
        pl.col('kWhDelivered').mean().alias('avg_kWh'),
    ])
    .sort('total_kWh', descending=True)
    .head(10)
)

---
## 4. Polars — 대용량 파일 sink (결과를 파일로 저장)

메모리에 올리지 않고 **변환 결과를 바로 Parquet으로 저장**합니다.  
Parquet으로 변환해두면 이후 탐색이 훨씬 빠릅니다 (컬럼형 저장 + 압축).

In [ ]:
# ── 4-1. 시계열 컬럼 제외 → Parquet 저장 (메모리 사용 최소) ───────────────
OUTPUT_PARQUET = f"{DATA_DIR}/office001_meta.parquet"

(
    pl.scan_csv(files['office001'], encoding='utf8-lossy')
    .select([
        '_id', 'userID', 'sessionID', 'stationID', 'spaceID',
        'siteID', 'clusterID', 'connectionTime', 'disconnectTime',
        'kWhDelivered', 'doneChargingTime', 'timezone'
    ])
    .sink_parquet(OUTPUT_PARQUET, compression='zstd')  # streaming write
)

parquet_size = os.path.getsize(OUTPUT_PARQUET) / 1024**2
print(f"Parquet 저장 완료: {OUTPUT_PARQUET} ({parquet_size:.1f} MB)")

In [ ]:
# ── 4-2. Parquet으로 재로드 (이후 탐색은 이것만 사용) ─────────────────────
df_pq = pl.read_parquet(OUTPUT_PARQUET)
print(df_pq.shape)
df_pq.head()

---
## 5. Pandas chunksize — 특정 컬럼 파싱이 필요할 때

`pilotSignal` / `chargingCurrent` 컬럼의 시계열 데이터를 직접 파싱해야 할 때 사용합니다.

In [ ]:
# ── 5-1. 청크 단위로 읽어서 집계 ─────────────────────────────────────────
CHUNK = 100  # 행이 적어도 용량이 크므로 청크 작게

results = []
reader = pd.read_csv(
    files['office001'],
    encoding='utf-8-sig',
    chunksize=CHUNK,
    usecols=['_id', 'stationID', 'kWhDelivered', 'connectionTime']  # 필요한 컬럼만
)

for chunk in reader:
    results.append(chunk.groupby('stationID')['kWhDelivered'].sum())

total_by_station = pd.concat(results).groupby(level=0).sum().sort_values(ascending=False)
total_by_station.head(10)

In [ ]:
# ── 5-2. 시계열 데이터 파싱 예시 (특정 세션 1개만) ───────────────────────
import ast
import matplotlib.pyplot as plt

# 첫 번째 행만 읽어서 chargingCurrent 파싱
sample = pd.read_csv(
    files['office001'],
    encoding='utf-8-sig',
    nrows=1,
    usecols=['_id', 'sessionID', 'chargingCurrent']
)

raw = sample['chargingCurrent'].iloc[0]
print("원본 문자열 앞 200자:")
print(raw[:200])

---
## 6. 전체 방법 비교 요약

| 방법 | 메모리 | 속도 | 추천 용도 |
|------|--------|------|----------|
| **DuckDB SQL** | 최소 (스트리밍) | 매우 빠름 | 집계, 필터, 다중 파일 쿼리 |
| **Polars scan_csv** | 최소 (lazy) | 빠름 | 컬럼 선택, 변환, Parquet 변환 |
| **Polars sink_parquet** | 최소 | — | 한 번 변환해두면 이후 탐색 극적으로 빠름 |
| **Pandas chunksize** | 적음 | 느림 | 시계열 파싱 등 행 단위 처리 |

**권장 워크플로우:**
1. DuckDB로 빠른 탐색 및 통계
2. Polars `sink_parquet`으로 시계열 컬럼 제외한 메타 데이터 변환 저장
3. 이후 모든 분석은 Parquet 파일로